# Prepare one Sentinel-2 acquisition for P3 inference

Notebook này audit và stage đúng **một** GeoTIFF sáu band cho checkpoint P3 T=1.
Nó không train, không load checkpoint và không sử dụng nhãn. Mặc định bên dưới
là ảnh Thái Thụy 21/09/2026 để so sánh trực tiếp P3 T=1 với P2 T=6.

## 1. Mount Drive và cài thư viện kiểm tra raster

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy==2.3.5', 'rasterio==1.5.2', 'pandas==2.2.3', 'matplotlib>=3.8,<4'
])
print('If NumPy was replaced in this runtime, restart the session once before continuing.')


## 2. Khóa dataset single-date

In [ ]:
from pathlib import Path

PROTOCOL = 'B_spatial_transfer'
PROTOCOL_FOLDER = 'protocol_B'
DATASET_ID = 'thai_thuy_2026_sep21_single'
REGION_ID = 'thai_thuy'
SOURCE_CONTAINER = Path(
    '/content/drive/MyDrive/Prithvi_Colab/data/sentinel2_exports/'
    'Sentinel2_ThaiThuy_2026_6bands_10m'
)
SOURCE_IMAGE = {
    'date': '2026-09-21',
    'filename': 'Sentinel2_2026-09-21_S2C_MSIL2A_20260921T_HSTDNNTHAITHUY_6bands_10m.tif',
    'stac_item_id': 'S2C_MSIL2A_20260921T031521_R118_T48QXH_20260921T082912',
}
TARGET_DATE = SOURCE_IMAGE['date']
OUTPUT_ROOT = (
    Path('/content/drive/MyDrive/Prithvi_Colab/data/inference_single')
    / PROTOCOL_FOLDER / DATASET_ID
)

assert PROTOCOL in {'A_temporal_transfer', 'B_spatial_transfer'}
assert Path(DATASET_ID).name == DATASET_ID
assert 'mosaic' not in SOURCE_IMAGE['filename'].lower()
print('Source:', SOURCE_CONTAINER)
print('Output:', OUTPUT_ROOT)


## 3. Materialize source từ folder hoặc ZIP

In [ ]:
import hashlib
import zipfile

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

filename = SOURCE_IMAGE['filename']
if SOURCE_CONTAINER.is_dir():
    source_path = SOURCE_CONTAINER / filename
    if not source_path.is_file():
        raise FileNotFoundError(source_path)
elif SOURCE_CONTAINER.is_file() and zipfile.is_zipfile(SOURCE_CONTAINER):
    staging = Path('/content') / f'sentinel2_single_{DATASET_ID}'
    staging.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SOURCE_CONTAINER) as archive:
        matches = [name for name in archive.namelist() if Path(name).name == filename]
        if len(matches) != 1:
            raise ValueError(f'Expected one ZIP member named {filename}, found {len(matches)}')
        source_path = staging / filename
        with archive.open(matches[0]) as source, open(source_path, 'wb') as target:
            for block in iter(lambda: source.read(1024 * 1024), b''):
                target.write(block)
else:
    raise FileNotFoundError('SOURCE_CONTAINER must be a folder or ZIP')

source_sha256 = sha256_file(source_path)
print(source_path.name, source_sha256)


## 4. Audit và stage GeoTIFF không đổi DN

In [ ]:
import json
import numpy as np
import rasterio

BANDS = ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']
rasters_dir = OUTPUT_ROOT / 'rasters'
rasters_dir.mkdir(parents=True, exist_ok=True)
output_path = rasters_dir / f"Sentinel2_{TARGET_DATE}_{REGION_ID}_6bands_10m.tif"

with rasterio.open(source_path) as source:
    if source.count != 6 or list(source.descriptions) != BANDS:
        raise ValueError('Wrong six-band contract')
    if source.tags().get('acquisition_date') != TARGET_DATE:
        raise ValueError('acquisition_date tag differs')
    if source.tags().get('source') != 'Microsoft Planetary Computer - Sentinel-2 L2A':
        raise ValueError('Unexpected source tag')
    if source.crs is None or not source.crs.is_projected:
        raise ValueError('A projected CRS is required')
    if not (
        np.isclose(abs(source.transform.a), 10.0)
        and np.isclose(abs(source.transform.e), 10.0)
        and np.isclose(source.transform.b, 0.0)
        and np.isclose(source.transform.d, 0.0)
    ):
        raise ValueError('GeoTIFF is not on a north-up 10 m grid')
    if list(source.dtypes) != ['uint16'] * 6 or source.nodata != 0:
        raise ValueError('Expected uint16 raw DN with nodata=0')
    array = source.read()
    valid = (source.read_masks() > 0).all(0) & np.isfinite(array).all(0)
    if not valid.any():
        raise ValueError('No valid pixel')
    array[:, ~valid] = 0
    profile = source.profile.copy()
    source_tags = source.tags()
    height, width = source.height, source.width
    target_crs = str(source.crs)
    transform = source.transform

if output_path.exists():
    with rasterio.open(output_path) as existing:
        if existing.tags().get('source_sha256') != source_sha256:
            raise FileExistsError('Existing output uses different source bytes; choose a new DATASET_ID')
else:
    profile.update(compress='deflate', tiled=True, nodata=0)
    output_tags = dict(source_tags)
    output_tags.update(
        acquisition_date=TARGET_DATE,
        stac_item_id=SOURCE_IMAGE['stac_item_id'],
        source_sha256=source_sha256,
        preparation_note='Single-date P3 input; pixel DN unchanged; reader offset not applied here',
    )
    temporary_path = output_path.with_name(output_path.stem + '.tmp.tif')
    with rasterio.open(temporary_path, 'w', **profile) as destination:
        destination.write(array)
        destination.write_mask(valid.astype(np.uint8) * 255)
        for index, description in enumerate(BANDS, start=1):
            destination.set_band_description(index, description)
        destination.update_tags(**output_tags)
    temporary_path.replace(output_path)

output_sha256 = sha256_file(output_path)
print('Staged:', output_path)
print('Shape:', height, width, '| valid pixels:', int(valid.sum()))
print('SHA-256:', output_sha256)


## 5. Visual QA và ghi manifest T=1

In [ ]:
import matplotlib.pyplot as plt

rgb = array[[2, 1, 0]].astype(np.float32)
for channel in range(3):
    values = rgb[channel][valid]
    low, high = np.percentile(values, [2, 98])
    rgb[channel] = np.clip((rgb[channel] - low) / max(high - low, 1), 0, 1)
rgb[:, ~valid] = 0
plt.figure(figsize=(10, 8))
plt.imshow(np.moveaxis(rgb, 0, -1))
plt.title(f'{REGION_ID} single-date input {TARGET_DATE}')
plt.axis('off')
plt.show()

manifest = {
    'schema': 'prithvi-inference-dataset-v1',
    'model_input_frames': 1,
    'protocol': PROTOCOL,
    'dataset_id': DATASET_ID,
    'region_id': REGION_ID,
    'target_date': TARGET_DATE,
    'images': [{
        'path': str(output_path.relative_to(OUTPUT_ROOT)).replace('\\', '/'),
        'date': TARGET_DATE,
        'sha256': output_sha256,
    }],
    'bands': BANDS,
    'grid': {
        'crs': target_crs,
        'shape': [height, width],
        'resolution_metres': 10,
        'same_region_as_training': PROTOCOL == 'A_temporal_transfer',
    },
    'radiometry': {
        'dn_offset': -1000.0,
        'scale': 10000.0,
        'offset_verified_for_source_product': True,
        'verification_note': (
            'Sentinel_2.ipynb writes individual Planetary Computer L2A spectral values '
            'without applying the P3 reader offset; the inference reader applies -1000 once.'
        ),
    },
    'normalization': {
        'mean': [1087, 1342, 1433, 2734, 1958, 1363],
        'std': [2248, 2179, 2178, 1850, 1242, 1049],
    },
    'inference': {
        'patch_size': 224,
        'stride': 112,
        'overlap_blending': 'mean_softmax_probability',
    },
    'evaluation': {
        'date_matched_ground_truth': None,
        'status': 'qualitative until an independent date-matched label set is supplied',
    },
    'provenance': {
        'source_container': str(SOURCE_CONTAINER),
        'source_notebook': 'Sentinel_2.ipynb individual-item export',
        'stac_collection': 'sentinel-2-l2a',
        'selected_item_ids': [SOURCE_IMAGE['stac_item_id']],
        'source_filename': filename,
        'source_sha256': source_sha256,
    },
}
manifest_path = OUTPUT_ROOT / 'inference_dataset.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Manifest:', manifest_path)
print('Manifest SHA-256:', sha256_file(manifest_path))
